![IITIS](pictures/logoIITISduze.png)

# Algorytmu wyczerpującego przeszukiwania

Inna nazwa to brute force. Tutaj użyjemy potężniejszych wersji tego algorytmu. Ogólna ide jest 


## Implementacja naiwna

W naiwnej implementacji wyczerpującego przeszukiwania


In [4]:
import numpy as np
from math import inf
from itertools import product
from tqdm import tqdm

# Jak działa product

# Funkcja pomocnicza - liczenie energii
def calculate_energy(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    return state @ J @ state.T + state @ h 



In [ ]:
# Ćwiczenie - sprubuj samodzielnie zaimplementować wyszukiwanie naiwne

def brute_force_naive(J, h):
    best_energy = inf
    best_state = None

    # główna pętla
    ...
    
    return best_energy, best_state

In [2]:
# Przykładowa odpowiedź

def brute_force_naive(J, h):
    best_energy = inf
    best_state = None
    n = len(h)

    for state in tqdm(product([-1, 1], repeat=n), desc="Wyczerpujące przeszukiwanie", total=2**n):
        state = np.array(state)
        energy = calculate_energy(J, h, state)
        if energy < best_energy:
            best_energy = energy
            best_state = state

    return best_energy, best_state

In [ ]:
# instancja testowa 16 spinów



In [4]:
# Polecam zobaczyć jak rozmiar n wpływa na czas obliczeń, dla n>20 czas obliczeń zaczyna rosnąć bardzo szybko
n = 16
rng = np.random.default_rng(seed=42)


scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor

sol_energy, sol_state = brute_force_naive(J, h)
print(sol_energy, sol_state)
print(J)
print(h)

Wyczerpujące przeszukiwanie: 100%|██████████| 65536/65536 [00:00<00:00, 158477.20it/s]

-24.497721731285164 [-1  1 -1 -1  1  1  1  1  1 -1  1  1 -1 -1 -1  1]
[[ 3.83847869e-01  4.12163377e-01 -7.42944841e-01 -7.61732834e-01
   8.34838884e-01 -1.66574946e-01 -8.01712325e-01  5.31228215e-01
   1.97374365e-01  1.82896134e-01 -7.78412586e-01  6.71070072e-01
  -9.67580730e-01 -7.90490469e-01 -3.65612672e-01 -6.76163413e-01]
 [ 0.00000000e+00 -9.56529148e-01  7.62428179e-02  7.13839046e-01
   5.18577095e-01 -8.01255167e-01  1.23780428e-01 -8.21050444e-01
  -4.84912965e-01  7.09994041e-02 -4.34039793e-01 -5.43045494e-01
   2.46440467e-01  4.19589447e-02 -9.92703246e-01  4.15745249e-01]
 [ 0.00000000e+00  0.00000000e+00  8.89063608e-01  5.64569313e-01
   1.94001184e-01  4.88491519e-01 -2.96691231e-01  6.47059329e-01
  -6.86554295e-01  8.08832952e-01  2.68921679e-01  1.28623460e-01
  -4.81861925e-01 -9.10021149e-01  5.38464607e-01  4.16680224e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.69445080e-02
   3.62551222e-02  2.32564908e-01  7.88520932e-01 -6.02268426e-01
   

Jak widzać ta implementacja nie jest zbyt efektywna

## Podstawy obliczeń równoległych na CPU

In [7]:
from itertools import product
from threading import Thread, Lock
from tqdm import tqdm
import numpy as np
from math import inf


def brute_force_naive_multithreaded(J, h, num_threads=4):
    best_energy = inf
    best_state = None
    n = len(h)

    lock = Lock()

    def process_states(states_chunk):
        nonlocal best_energy, best_state
        for state in states_chunk:
            state = np.array(state)
            energy = calculate_energy(J, h, state)
            with lock:
                if energy < best_energy:
                    best_energy = energy
                    best_state = state


    all_states = list(product([-1, 1], repeat=n))

    chunk_size = len(all_states) // num_threads
    chunks = [all_states[i:i + chunk_size] for i in range(0, len(all_states), chunk_size)]
   
    if len(chunks) > num_threads:
        chunks[-2].extend(chunks[-1])
        chunks.pop()

    threads = []
    for chunk in chunks:
        thread = Thread(target=process_states, args=(chunk,))
        threads.append(thread)
        thread.start()


    for thread in threads:
        thread.join()

    return best_energy, best_state

In [13]:
n = 16

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor


from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)

s, e = brute_force_naive_multithreaded(J, h)
print(s)

   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15     energy num_oc.
0 +1 -1 -1 -1 -1 -1 -1 +1 +1 +1 -1 -1 -1 +1 -1 +1 -28.334681       1
['SPIN', 1 rows, 1 samples, 16 variables]


Multithreaded exhaustive search:   0%|          | 0/65536 [00:00<?, ?it/s]

-28.334681423682497
